# Visualize WFS Data for Scope Regions

**Goal**: Fetch raw WFS geometries for each scope region and save them to the GeoPackage for visualization in QGIS.

**Why**: The `DataHandler` only keeps aggregated features (e.g., "majority crop type"). We want to see the actual geometries returned by WFS services.

**Strategy**:
1. Load scope regions from `phase1_2025-08-14_v1.gpkg`
2. For each scope region, use `DataCollector` to fetch WFS data
3. Combine all geometries into single layers per WFS service/layer
4. Save to GeoPackage with naming: `wfs_{service_name}_{layer_name}`
5. Open in QGIS and explore!

## Setup

In [31]:
import sys
sys.path.append("../../")

import src.paths as PATHS
import src.data.data_collector as DC
import src.data.config as DATA_CONFIG

import geopandas as gpd
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime
import shutil



print("✅ Setup complete!")

✅ Setup complete!


## 1. Load Scope Regions

Using the most recent data: `phase1_2025-08-14_v1.gpkg`

In [42]:
# Path to the most recent GeoPackage
filename = "wocu_output_fase2_v4"
gpkg_path = PATHS.DATA_DIR / f"{filename}.gpkg"


# Get current date in YYYYMMDD format
date_suffix = datetime.now().strftime("%Y%m%d")
new_path = PATHS.DATA_DIR / f"{filename}_w_features_{date_suffix}.gpkg"
shutil.copy2(gpkg_path, new_path)

# Load scope regions
scope_regions = gpd.read_file(new_path, layer="vlakken_scope")

print(f"📍 Loaded {len(scope_regions)} scope regions")
print(f"📐 CRS: {scope_regions.crs}")
print(f"\n🗺️ Columns: {list(scope_regions.columns)}")
scope_regions.head()

📍 Loaded 12130 scope regions
📐 CRS: EPSG:28992

🗺️ Columns: ['position_id', 'waterlichaam', 'geometry']


,position_id,waterlichaam,geometry
0,rijn_l_10343_10350,"Boven-Rijn, Waal, Boven-Merwede, Beneden-Merwe...","POLYGON ((119484.028 425575.146, 119552 425573..."
1,rijn_r_10343_10350,"Boven-Rijn, Waal, Boven-Merwede, Beneden-Merwe...","POLYGON ((119552 425573, 119484.028 425575.146..."
2,rijn_l_10350_10360,"Boven-Rijn, Waal, Boven-Merwede, Beneden-Merwe...","POLYGON ((119384.07 425578.303, 119484.028 425..."
3,rijn_r_10350_10360,"Boven-Rijn, Waal, Boven-Merwede, Beneden-Merwe...","POLYGON ((119484.028 425575.146, 119384.07 425..."
4,rijn_l_10360_10370,"Boven-Rijn, Waal, Boven-Merwede, Beneden-Merwe...","POLYGON ((119284.111 425581.46, 119384.07 4255..."


## 2. Set up WFS Configuration

We'll use the default configuration which includes:
- Land Use (BRP Gewaspercelen)
- Buildings (BAG)
- Vegetation Legger (3 layers: bomen, heggen, vegetatieklassen)

In [43]:
# Load the default configuration
config = DATA_CONFIG.DataConfiguration()

print("📡 WFS Services to query:")
for i, wfs_service in enumerate(config.known_wfs_services, 1):
    print(f"\n{i}. {wfs_service.name}")
    print(f"   URL: {wfs_service.url}")
    print(f"   Layers: {', '.join(wfs_service.relevant_layers)}")

📡 WFS Services to query:

1. land_use
   URL: https://service.pdok.nl/rvo/brpgewaspercelen/wfs/v1_0
   Layers: BrpGewas

2. building_location
   URL: https://service.pdok.nl/lv/bag/wfs/v2_0
   Layers: bag:pand

3. vegetation
   URL: https://geo.rijkswaterstaat.nl/services/ogc/gdr/rws_vegetatielegger/ows?version=2.0.0
   Layers: rws_vegetatielegger:bomen, rws_vegetatielegger:heggen, rws_vegetatielegger:vegetatieklassen


## 3. Helper Function to Save WFS Data

This function will:
- Generate a clean layer name: `wfs_{service_name}_{layer_name}`
- Combine geometries from all scope regions
- Save to the GeoPackage

In [44]:
def save_wfs_layer_to_gpkg(
    wfs_data_list: list,
    service_name: str,
    layer_name: str,
    output_path: Path,
    scope_region_ids: list = None,
):
    """
    Combine WFS data from multiple scope regions and save to GeoPackage.
    
    Handles duplicates: Large features (e.g., vegetation zones) that span multiple
    scope regions will be fetched multiple times. We deduplicate based on geometry.
    
    Args:
        wfs_data_list: List of GeoDataFrames (one per scope region)
        service_name: Name of the WFS service (e.g., 'land_use')
        layer_name: Name of the WFS layer (e.g., 'BrpGewas')
        output_path: Path to the output GeoPackage
        scope_region_ids: Optional list of scope region IDs to add as a column
    """
    if not wfs_data_list:
        print(f"⚠️  No data to save for {service_name}:{layer_name}")
        return
    
    # Add scope_region_id to each dataframe if provided
    if scope_region_ids and len(scope_region_ids) == len(wfs_data_list):
        for gdf, region_id in zip(wfs_data_list, scope_region_ids):
            gdf['scope_region_id'] = region_id
    
    # Combine all dataframes
    combined = pd.concat(wfs_data_list, ignore_index=True)
    original_count = len(combined)
    
    # Remove duplicates based on geometry
    # Use WKT (Well-Known Text) representation for comparison
    combined['_geom_wkt'] = combined.geometry.apply(lambda x: x.wkt)
    
    # Keep first occurrence, track which scope_region_id it came from
    combined_dedup = combined.drop_duplicates(subset=['_geom_wkt'], keep='first')
    combined_dedup = combined_dedup.drop(columns=['_geom_wkt'])
    
    duplicates_removed = original_count - len(combined_dedup)
    
    # Generate layer name
    # Clean up service name (remove special chars, spaces)
    clean_service = service_name.replace(' ', '_').replace(':', '_').lower()
    clean_layer = layer_name.replace(':', '_').replace(' ', '_')
    output_layer_name = f"wfs_{clean_service}_{clean_layer}"
    
    # Save to GeoPackage
    combined_dedup.to_file(output_path, layer=output_layer_name, driver="GPKG")
    
    if duplicates_removed > 0:
        print(f"✅ Saved {len(combined_dedup)} features to layer: {output_layer_name}")
        print(f"   🔄 Removed {duplicates_removed} duplicates ({duplicates_removed/original_count*100:.1f}%)")
    else:
        print(f"✅ Saved {len(combined_dedup)} features to layer: {output_layer_name}")
    
    return output_layer_name

print("✅ Helper function defined (with duplicate removal)!")

✅ Helper function defined (with duplicate removal)!


## 4. Helper Function: Fetch WFS Data

In [45]:
def fetch_wfs_data_for_regions(
    scope_regions: gpd.GeoDataFrame,
    wfs_services: list,
    buffer_in_metres: float = 10,
    wfs_timeout: int = 30,
    max_retries: int = 3,
    test_mode: bool = False,
    num_test_regions: int = 10,
):
    """
    Fetch WFS data for multiple scope regions.
    
    Args:
        scope_regions: GeoDataFrame with scope regions (polygons) to query
        wfs_services: List of WfsService objects to query
        buffer_in_metres: Buffer around each region (default: 10m)
        wfs_timeout: Timeout per WFS request (default: 30s)
        max_retries: Retry attempts per service (default: 3)
        test_mode: If True, only process first N regions (default: False)
        num_test_regions: Number of regions to process in test mode (default: 10)
    
    Returns:
        tuple: (wfs_data_collection, scope_region_ids, successful_regions, failed_regions)
    """
    # Determine which regions to process
    if test_mode:
        regions_to_process = scope_regions.head(num_test_regions)
        num_regions = num_test_regions
    else:
        regions_to_process = scope_regions
        num_regions = len(scope_regions)
    
    print(f"🔄 Processing {num_regions} scope regions...")
    print(f"⏱️  DataCollector: {wfs_timeout}s timeout, {max_retries} retries per service")
    if test_mode:
        print(f"⚠️  TEST MODE: Processing only first {num_regions} regions")
    print()
    
    # Initialize collections
    wfs_data_collection = {}
    scope_region_ids = []
    successful_regions = []
    failed_regions = []
    
    # Loop through each scope region
    for idx, row in tqdm(regions_to_process.iterrows(), total=num_regions, desc="Fetching WFS data"):
        region_geom = row.geometry
        region_id = row.get('location_id', f"region_{idx}")
        scope_region_ids.append(region_id)
        
        try:
            # Create DataCollector with configuration
            data_collector = DC.DataCollector(
                source_shape=region_geom,
                source_epsg_crs=scope_regions.crs.to_epsg(),
                buffer_in_metres=buffer_in_metres,
                wfs_services=wfs_services,
                wfs_timeout=wfs_timeout,
                max_retries=max_retries,
            )
            
            # Fetch data from all WFS services
            data_collector.get_data_from_all_wfs()
            
            # Extract the data from the nested dictionary
            for service_name, layers_dict in data_collector.relevant_geospatial_data.items():
                if service_name not in wfs_data_collection:
                    wfs_data_collection[service_name] = {}
                
                for layer_name, gdf in layers_dict.items():
                    if layer_name not in wfs_data_collection[service_name]:
                        wfs_data_collection[service_name][layer_name] = []
                    
                    # Add this region's data to the collection
                    if gdf is not None and len(gdf) > 0:
                        wfs_data_collection[service_name][layer_name].append(gdf.copy())
            
            successful_regions.append(region_id)
        
        except Exception as e:
            error_type = type(e).__name__
            print(f"\n❌ Region {region_id} failed: {error_type}")
            failed_regions.append((region_id, str(e)[:100]))
            continue
    
    # Print summary
    print(f"\n✅ Data collection complete!")
    print(f"   - Successful: {len(successful_regions)}/{num_regions}")
    print(f"   - Failed: {len(failed_regions)}/{num_regions}")
    
    if failed_regions:
        print(f"\n⚠️  Failed regions ({len(failed_regions)}):")
        for region_id, error in failed_regions[:5]:  # Show first 5
            print(f"   - {region_id}: {error[:50]}")
        if len(failed_regions) > 5:
            print(f"   ... and {len(failed_regions) - 5} more")
    
    print(f"\n📊 Summary:")
    for service_name, layers_dict in wfs_data_collection.items():
        print(f"\n  {service_name}:")
        for layer_name, gdf_list in layers_dict.items():
            total_features = sum(len(gdf) for gdf in gdf_list)
            print(f"    - {layer_name}: {len(gdf_list)} regions, {total_features} total features")
    
    return wfs_data_collection, scope_region_ids, successful_regions, failed_regions

print("✅ Function defined!")

✅ Function defined!


## 5. Fetch Production WFS Data (All 233 Regions)

Using the function above, fetch data from the 3 production WFS services.

In [46]:
# Fetch WFS data for all 233 scope regions using production services
wfs_data_collection, scope_region_ids, successful_regions, failed_regions = fetch_wfs_data_for_regions(
    scope_regions=scope_regions,
    wfs_services=config.known_wfs_services,
    buffer_in_metres=config.prediction_region_buffer,
    wfs_timeout=30,
    max_retries=3,
    test_mode=False,  # Set to True to test with 10 regions first
    num_test_regions=10,
)

🔄 Processing 12130 scope regions...
⏱️  DataCollector: 30s timeout, 3 retries per service



Fetching WFS data:  22%|██▏       | 2714/12130 [1:36:06<5:33:27,  2.12s/it] 


KeyboardInterrupt: 

## 5. Save to GeoPackage

In [27]:
print("💾 Saving WFS data to GeoPackage...\n")

saved_layers = []

for service_name, layers_dict in wfs_data_collection.items():
    print(f"\n📡 Service: {service_name}")
    
    for layer_name, gdf_list in layers_dict.items():
        layer_output_name = save_wfs_layer_to_gpkg(
            wfs_data_list=gdf_list,
            service_name=service_name,
            layer_name=layer_name,
            output_path=new_path,
            scope_region_ids=scope_region_ids[:len(gdf_list)],
        )
        if layer_output_name:
            saved_layers.append(layer_output_name)

print(f"\n\n🎉 DONE! Saved {len(saved_layers)} new layers to {gpkg_path.name}")
print(f"\n📋 New layers:")
for layer in saved_layers:
    print(f"  - {layer}")
print(f"\n👉 Open {gpkg_path.name} in QGIS to visualize!")

💾 Saving WFS data to GeoPackage...


📡 Service: land_use
✅ Saved 236 features to layer: wfs_land_use_BrpGewas
   🔄 Removed 321 duplicates (57.6%)

📡 Service: building_location
✅ Saved 117 features to layer: wfs_building_location_bag_pand
   🔄 Removed 23 duplicates (16.4%)

📡 Service: vegetation
✅ Saved 246 features to layer: wfs_vegetation_rws_vegetatielegger_bomen
   🔄 Removed 88 duplicates (26.3%)
✅ Saved 26 features to layer: wfs_vegetation_rws_vegetatielegger_heggen
   🔄 Removed 2 duplicates (7.1%)
✅ Saved 667 features to layer: wfs_vegetation_rws_vegetatielegger_vegetatieklassen
   🔄 Removed 1296 duplicates (66.0%)


🎉 DONE! Saved 5 new layers to phase1_2025-08-14_v1.gpkg

📋 New layers:
  - wfs_land_use_BrpGewas
  - wfs_building_location_bag_pand
  - wfs_vegetation_rws_vegetatielegger_bomen
  - wfs_vegetation_rws_vegetatielegger_heggen
  - wfs_vegetation_rws_vegetatielegger_vegetatieklassen

👉 Open phase1_2025-08-14_v1.gpkg in QGIS to visualize!


## 6. Verify: List All Layers in GeoPackage

In [10]:
# List all layers in the GeoPackage
# GeoPackage is SQLite, so we can query it directly
import sqlite3

conn = sqlite3.connect(gpkg_path)
cursor = conn.cursor()

# Query the gpkg_contents table which lists all layers
cursor.execute("SELECT table_name, data_type FROM gpkg_contents ORDER BY table_name")
layers = cursor.fetchall()
conn.close()

print(f"📦 All layers in {gpkg_path.name}:\n")

# Separate original layers from WFS layers
original_layers = [(name, dtype) for name, dtype in layers if not name.startswith('wfs_')]
wfs_layers = [(name, dtype) for name, dtype in layers if name.startswith('wfs_')]

print(f"📂 Original layers ({len(original_layers)}):")
for layer_name, data_type in original_layers:
    print(f"  - {layer_name} ({data_type})")

print(f"\n🆕 WFS layers ({len(wfs_layers)}):")
for layer_name, data_type in wfs_layers:
    print(f"  - {layer_name} ({data_type})")

📦 All layers in phase1_2025-08-14_v1.gpkg:

📂 Original layers (6):
  - beschermde_oever (features)
  - middenlijn (features)
  - punten_oever (features)
  - summary_layer (features)
  - vlakken_erosie (features)
  - vlakken_scope (features)

🆕 WFS layers (5):
  - wfs_building_location_bag_pand (features)
  - wfs_land_use_BrpGewas (features)
  - wfs_vegetation_rws_vegetatielegger_bomen (features)
  - wfs_vegetation_rws_vegetatielegger_heggen (features)
  - wfs_vegetation_rws_vegetatielegger_vegetatieklassen (features)


---

## ✅ Next Steps

1. **Open in QGIS**: Load `phase1_2025-08-14_v1.gpkg` and explore the `wfs_*` layers
2. **Verify data quality**: 
   - Do the geometries make sense? 
   - Are they in the right locations?
   - Do the attributes look correct?
3. **Compare with scope regions**: Layer the WFS data over `vlakken_scope` to see spatial relationships
4. **Decide on integration**: If this works well, we can add a method to `DataCollector` to save raw WFS data
5. **Test new services**: Next, try this same approach with RWS Legger and BKN

**Expected layers created:**
- `wfs_land_use_BrpGewas` - Agricultural crop parcels
- `wfs_building_location_bag_pand` - Building footprints  
- `wfs_vegetation_rws_vegetatielegger_bomen` - Trees
- `wfs_vegetation_rws_vegetatielegger_vegetatieklassen` - Vegetation classes
- (possibly) `wfs_vegetation_rws_vegetatielegger_heggen` - Hedges

---

## 7. Explore New Service: RWS Legger (All 46 Layers)

**Goal**: Test ALL 46 layers from RWS Legger to identify which are useful for erosion prediction.

**Strategy**:
- Use `luke_inputs_v3.gpkg` (11 regions - faster testing)
- Fetch ALL 46 RWS Legger layers
- Save to same GeoPackage
- Open in QGIS to visually assess usefulness
- Then cherry-pick 3-5 layers for production use

**Why 11 regions?**
- Fast testing: 11 regions × 46 layers = ~506 requests (~10-15 min)
- vs. 233 regions × 46 layers = 10,718 requests (~90 hours)
- Same regions used in `demo_baseline_model.ipynb`

### 7.1 Load Luke's 11 Test Regions

In [11]:
# Load the same GeoPackage used in demo_baseline_model.ipynb
luke_gpkg_path = PATHS.DATA_DIR / "luke_inputs_v3.gpkg"
luke_scope_regions = gpd.read_file(luke_gpkg_path, layer="vlakken_scope")

print(f"📍 Loaded {len(luke_scope_regions)} test regions from {luke_gpkg_path.name}")
print(f"📐 CRS: {luke_scope_regions.crs}")
print(f"\n🗺️  Regions: {luke_scope_regions['location_id'].tolist()}")
luke_scope_regions.head()

📍 Loaded 11 test regions from luke_inputs_v3.gpkg
📐 CRS: EPSG:28992

🗺️  Regions: ['waal_949_0_949_1_left', 'waal_949_1_949_2_left', 'waal_949_2_949_3000000000001_left', 'waal_949_3_949_4_left', 'waal_949_4_949_5_left', 'waal_949_5_949_6_left', 'waal_949_6_949_7_left', 'waal_949_7_949_8000000000001_left', 'waal_949_8_949_9_left', 'waal_949_9_950_0_left', 'waal_950_0_950_1_left']


,location_id,start_year,end_year,geometry
0,waal_949_0_949_1_left,2015,2024,"POLYGON ((132009.938 425632.791, 132014.782 42..."
1,waal_949_1_949_2_left,2015,2024,"POLYGON ((131920.781 425583.484, 131925.953 42..."
2,waal_949_2_949_3000000000001_left,2015,2024,"POLYGON ((131827.145 425543.344, 131833.664 42..."
3,waal_949_3_949_4_left,2015,2024,"POLYGON ((131730.266 425511.855, 131732.629 42..."
4,waal_949_4_949_5_left,2015,2024,"POLYGON ((131632.419 425483.317, 131636.881 42..."


### 7.2 Define RWS Legger Service (All 46 Layers)

We'll query ALL layers to see what's available. In production, we'll only use 3-5 useful ones.

In [12]:
# Define RWS Legger service with ALL 46 layers
# We'll fetch everything, then decide in QGIS which ones are useful

from src.data.schema_wfs_service import WfsService
from owslib.wfs import WebFeatureService

# First, let's get the actual layer list from the service
rws_legger_url = "https://geo.rijkswaterstaat.nl/services/ogc/gdr/rws_legger/ows?version=2.0.0"
print(f"🔍 Connecting to RWS Legger to get layer list...")

wfs = WebFeatureService(rws_legger_url, version="2.0.0", timeout=30)
all_layers = list(wfs.contents.keys())

print(f"✅ Found {len(all_layers)} layers\n")
print("📋 Full layer list:")
for i, layer in enumerate(all_layers, 1):
    short_name = layer.split(':')[-1] if ':' in layer else layer
    print(f"  {i:2d}. {short_name}")

# Create WFS service with ALL layers
rws_legger_service = WfsService(
    name="rws_legger",
    url=rws_legger_url,
    relevant_layers=all_layers,
    version="2.0.0"
)

print(f"\n✅ RWS Legger service configured with {len(all_layers)} layers")

🔍 Connecting to RWS Legger to get layer list...
✅ Found 46 layers

📋 Full layer list:
   1. andere_dan_primaire_waterkering_beschermingszone_legger
   2. andere_dan_primaire_waterkering_waterstaatswerk_legger
   3. begrenzing_rijksvaarweg_legger
   4. bladkader_legger
   5. dam_legger
   6. duiker_lijn_legger
   7. duiker_vlak_legger
   8. dwarsprofiel_over_genormeerde_bodem_legger
   9. dwarsprofiel_over_primaire_kering_legger
  10. dwarsprofiel_over_regionale_kering_legger
  11. gemaal_legger
  12. genormeerd_bodem_legger
  13. in_of_uitwateringssluis_legger
  14. kilometrering_legger
  15. krib_legger
  16. kribhoogtes_legger
  17. kunstwerk_niet_in_beheer_bij_rws_legger
  18. lengteprofiel_over_primaire_kering_legger
  19. lengteprofiel_over_regionale_kering_legger
  20. natuurvriendelijke_oever_lijn_legger
  21. natuurvriendelijke_oever_vlak_legger
  22. natuurvriendelijke_vooroever_legger
  23. nevengeul_strang_legger
  24. niet_primaire_waterkering_beschermingszone__overig_en_re

### 7.3 Fetch RWS Legger Data for 11 Test Regions

This will take ~10-15 minutes depending on network speed and data volume.

In [13]:
# Fetch RWS Legger data for 11 test regions
rws_data_collection, rws_region_ids, rws_successful, rws_failed = fetch_wfs_data_for_regions(
    scope_regions=luke_scope_regions,
    wfs_services=[rws_legger_service],  # Only RWS Legger
    buffer_in_metres=10,
    wfs_timeout=30,
    max_retries=3,
    test_mode=False,  # Process all 11 regions
)

🔄 Processing 11 scope regions...
⏱️  DataCollector: 30s timeout, 3 retries per service



Fetching WFS data: 100%|██████████| 11/11 [02:37<00:00, 14.35s/it]


✅ Data collection complete!
   - Successful: 11/11
   - Failed: 0/11

📊 Summary:

  rws_legger:
    - rws_legger:andere_dan_primaire_waterkering_beschermingszone_legger: 0 regions, 0 total features
    - rws_legger:andere_dan_primaire_waterkering_waterstaatswerk_legger: 0 regions, 0 total features
    - rws_legger:begrenzing_rijksvaarweg_legger: 11 regions, 11 total features
    - rws_legger:bladkader_legger: 11 regions, 35 total features
    - rws_legger:dam_legger: 0 regions, 0 total features
    - rws_legger:duiker_lijn_legger: 0 regions, 0 total features
    - rws_legger:duiker_vlak_legger: 0 regions, 0 total features
    - rws_legger:dwarsprofiel_over_genormeerde_bodem_legger: 5 regions, 5 total features
    - rws_legger:dwarsprofiel_over_primaire_kering_legger: 0 regions, 0 total features
    - rws_legger:dwarsprofiel_over_regionale_kering_legger: 0 regions, 0 total features
    - rws_legger:gemaal_legger: 0 regions, 0 total features
    - rws_legger:genormeerd_bodem_legger: 11 

### 7.4 Save RWS Legger Layers to GeoPackage

In [20]:
print("💾 Saving RWS Legger data to GeoPackage...\n")

saved_rws_layers = []

for service_name, layers_dict in rws_data_collection.items():
    print(f"\n📡 Service: {service_name}")
    
    for layer_name, gdf_list in layers_dict.items():
        layer_output_name = save_wfs_layer_to_gpkg(
            wfs_data_list=gdf_list,
            service_name=service_name,
            layer_name=layer_name,
            output_path=luke_gpkg_path,
            scope_region_ids=rws_region_ids[:len(gdf_list)],
        )
        if layer_output_name:
            saved_rws_layers.append(layer_output_name)

print(f"\n\n🎉 DONE! Saved {len(saved_rws_layers)} RWS Legger layers to {luke_gpkg_path.name}")
print(f"\n📋 Saved layers ({len(saved_rws_layers)}):")
for layer in saved_rws_layers:
    print(f"  - {layer}")
print(f"\n👉 Open {luke_gpkg_path.name} in QGIS to explore!")

💾 Saving RWS Legger data to GeoPackage...


📡 Service: rws_legger
⚠️  No data to save for rws_legger:rws_legger:andere_dan_primaire_waterkering_beschermingszone_legger
⚠️  No data to save for rws_legger:rws_legger:andere_dan_primaire_waterkering_waterstaatswerk_legger
✅ Saved 1 features to layer: wfs_rws_legger_rws_legger_begrenzing_rijksvaarweg_legger
   🔄 Removed 10 duplicates (90.9%)
✅ Saved 4 features to layer: wfs_rws_legger_rws_legger_bladkader_legger
   🔄 Removed 31 duplicates (88.6%)
⚠️  No data to save for rws_legger:rws_legger:dam_legger
⚠️  No data to save for rws_legger:rws_legger:duiker_lijn_legger
⚠️  No data to save for rws_legger:rws_legger:duiker_vlak_legger
✅ Saved 2 features to layer: wfs_rws_legger_rws_legger_dwarsprofiel_over_genormeerde_bodem_legger
   🔄 Removed 3 duplicates (60.0%)
⚠️  No data to save for rws_legger:rws_legger:dwarsprofiel_over_primaire_kering_legger
⚠️  No data to save for rws_legger:rws_legger:dwarsprofiel_over_regionale_kering_legger
⚠️  No da

### 7.5 Next Steps: Visual Assessment in QGIS

**What to do now:**

1. **Open QGIS** and load `luke_inputs_v3.gpkg`
2. **Load all `wfs_rws_legger_*` layers** to see what each contains
3. **Assess usefulness** for erosion prediction:
   - Does it show banks, structures, or relevant features?
   - Is the data quality good?
   - Is it dense enough to be useful?
   
4. **Identify 3-5 useful layers** such as:
   - `oever_*` (shore/bank layers)
   - `krib_*` (groynes - erosion protection structures)
   - `nevengeul_*` (side channels)
   - NVO-related layers
   
5. **Update config** (`backend/src/config.py`) with only the useful layers
6. **Re-run Section 5** with updated config for all 233 regions

**Questions to answer:**
- Which layers have data in our test regions?
- Which layers are empty/sparse?
- Which layers correlate with erosion-prone areas?

---

## 8. Batch Download for All 233 Scope Regions

**Goal:** Download WFS data from RWS Legger and BKN Nature for all prediction regions and save to geopackage.

**Approach:** Use existing `DataCollector` pattern, loop through all regions with progress tracking.

In [8]:
### 8.1 Load All 233 Scope Regions

# Load the canonical geopackage with all regions
full_gpkg_path = PATHS.DATA_DIR / "phase1_2025-08-14_v1.gpkg"
all_scope_regions = gpd.read_file(full_gpkg_path, layer="vlakken_scope")

print(f"📍 Loaded {len(all_scope_regions)} scope regions")
print(f"📐 CRS: {all_scope_regions.crs}")
print(f"🗺️  Unique locations: {all_scope_regions['location_id'].nunique()}")
print(f"\nFirst few regions:")
all_scope_regions[['location_id', 'start_year', 'end_year', 'geometry']].head()

📍 Loaded 233 scope regions
📐 CRS: EPSG:28992
🗺️  Unique locations: 233

First few regions:


,location_id,start_year,end_year,geometry
0,maas_l_2180_2181,2016,2024,"POLYGON ((148604.881 416309.328, 148538.207 41..."
1,maas_l_2181_2182,2016,2024,"POLYGON ((148491.678 416329.661, 148416.977 41..."
2,maas_l_2182_2183,2016,2024,"POLYGON ((148373.399 416364.687, 148290.662 41..."
3,maas_l_2183_2184,2016,2024,"POLYGON ((148268.941 416410.637, 148182.141 41..."
4,maas_l_2184_2185,2016,2024,"POLYGON ((148182.141 416457.702, 148095.34 416..."


In [13]:
### 8.2 Auto-Discover ALL Layers (RWS Legger + BKN Nature)

import src.data.schema_wfs_service as SWS
from owslib.wfs import WebFeatureService

# Services to query
services_to_discover = [
    {
        "name": "rws_legger",
        "url": "https://geo.rijkswaterstaat.nl/services/ogc/gdr/rws_legger/ows",
        "version": "2.0.0"
    },
    {
        "name": "bkn_nature", 
        "url": "https://geo.rijkswaterstaat.nl/services/ogc/gdr/beheerkaart_nat/ows",
        "version": "2.0.0"
    },
]

new_wfs_services = []

print("🔍 Discovering layers from WFS services...\n")
print("="*80)

for service_info in services_to_discover:
    print(f"\n📡 {service_info['name'].upper()}")
    print(f"   URL: {service_info['url']}")
    
    try:
        # Connect to WFS
        wfs = WebFeatureService(service_info['url'], version=service_info['version'], timeout=30)
        
        # Get all available layers
        all_layers = list(wfs.contents.keys())
        
        print(f"   ✅ Found {len(all_layers)} layers:")
        print()
        
        # Show all layers with descriptions if available
        for i, layer_name in enumerate(all_layers, 1):
            # Try to get layer title/abstract for better understanding
            layer_obj = wfs.contents.get(layer_name)
            title = layer_obj.title if hasattr(layer_obj, 'title') else layer_name
            
            # Clean up layer name for display
            display_name = layer_name.split(':')[-1] if ':' in layer_name else layer_name
            
            print(f"      {i:2d}. {display_name}")
            if title and title != layer_name:
                print(f"          └─ {title}")
        
        # Create WfsService with ALL layers
        new_wfs_services.append(
            SWS.WfsService(
                name=service_info['name'],
                url=service_info['url'],
                version=service_info['version'],
                relevant_layers=all_layers,
            )
        )
        
        print(f"\n   📦 Will download data from ALL {len(all_layers)} layers")
        
    except Exception as e:
        print(f"   ❌ Failed to connect: {e}")
        print(f"   Using empty layer list")
        new_wfs_services.append(
            SWS.WfsService(
                name=service_info['name'],
                url=service_info['url'],
                version=service_info['version'],
                relevant_layers=[],
            )
        )

print("\n" + "="*80)
print(f"\n✅ Ready to download from {len(new_wfs_services)} services")
print(f"   Total layers to test: {sum(len(s.relevant_layers) for s in new_wfs_services)}")

🔍 Discovering layers from WFS services...


📡 RWS_LEGGER
   URL: https://geo.rijkswaterstaat.nl/services/ogc/gdr/rws_legger/ows
   ✅ Found 46 layers:

       1. andere_dan_primaire_waterkering_beschermingszone_legger
          └─ andere_dan_primaire_waterkering_beschermingszone_legger
       2. andere_dan_primaire_waterkering_waterstaatswerk_legger
          └─ andere_dan_primaire_waterkering_waterstaatswerk_legger
       3. begrenzing_rijksvaarweg_legger
          └─ begrenzing_rijksvaarweg_legger
       4. bladkader_legger
          └─ bladkader_legger
       5. dam_legger
          └─ dam_legger
       6. duiker_lijn_legger
          └─ duiker_lijn_legger
       7. duiker_vlak_legger
          └─ duiker_vlak_legger
       8. dwarsprofiel_over_genormeerde_bodem_legger
          └─ dwarsprofiel_over_genormeerde_bodem_legger
       9. dwarsprofiel_over_primaire_kering_legger
          └─ dwarsprofiel_over_primaire_kering_legger
      10. dwarsprofiel_over_regionale_kering_legger
      

In [16]:
# Combine OLD (existing) + NEW (RWS Legger + BKN) services
import src.config as CONFIG
all_wfs_services = CONFIG.KNOWN_WFS_SERVICES + new_wfs_services

# Fetch data for all 233 regions from ALL services
all_data_collection, all_region_ids, all_successful, all_failed = fetch_wfs_data_for_regions(
    scope_regions=all_scope_regions,  # All 233 regions
    wfs_services=all_wfs_services,    # Combined list of ALL services
    buffer_in_metres=0,
    wfs_timeout=120,
    max_retries=2,
    test_mode=False,  # Process all regions
)

🔄 Processing 233 scope regions...
⏱️  DataCollector: 120s timeout, 2 retries per service



Fetching WFS data:   0%|          | 0/233 [00:00<?, ?it/s]

Fetching WFS data:  93%|█████████▎| 217/233 [1:52:11<09:01, 33.85s/it]Failed to get data from rws_legger (attempt 1/2): HTTPSConnectionPool(host='geo.rijkswaterstaat.nl', port=443): Read timed out. (read timeout=120). Retrying...
Failed to get data from rws_legger after 2 attempts: 502 Server Error: Bad Gateway for url: https://geo.rijkswaterstaat.nl/services/ogc/gdr/rws_legger/wfs?service=WFS&version=2.0.0&request=GetFeature&bbox=148674.2426924839%2C416272.03256576386%2C148775.86101238837%2C416332.4883459758%2Curn%3Aogc%3Adef%3Acrs%3AEPSG%3A%3A28992&typenames=rws_legger%3Aandere_dan_primaire_waterkering_beschermingszone_legger&count=10000&outputFormat=json
Failed to get data from bkn_nature (attempt 1/2): 502 Server Error: Bad Gateway for url: https://geo.rijkswaterstaat.nl/services/ogc/gdr/beheerkaart_nat/wfs?service=WFS&version=2.0.0&request=GetFeature&bbox=148674.2426924839%2C416272.03256576386%2C148775.86101238837%2C416332.4883459758%2Curn%3Aogc%3Adef%3Acrs%3AEPSG%3A%3A28992&typen


❌ Region nevengeul_ijssel_l_0050_0051 failed: HTTPError


Fetching WFS data:  94%|█████████▍| 220/233 [1:54:21<06:40, 30.80s/it]

...


❌ Region nevengeul_ijssel_l_0026_0027 failed: HTTPError


Fetching WFS data: 100%|█████████▉| 232/233 [1:54:25<00:00,  1.30it/s]


❌ Region rijn_r_9482_9483 failed: HTTPError


Fetching WFS data: 100%|██████████| 233/233 [1:54:26<00:00, 29.47s/it]


❌ Region ijssel_r_9479_9480 failed: HTTPError

✅ Data collection complete!
   - Successful: 218/233
   - Failed: 15/233

⚠️  Failed regions (15):
   - nevengeul_ijssel_l_0050_0051: 502 Server Error: Bad Gateway for url: https://geo
   - ijssel_l_9486_9487: 502 Server Error: Bad Gateway for url: https://geo
   - nevengeul_rijn_l_0214_0215: 502 Server Error: Bad Gateway for url: https://geo
   - nevengeul_ijssel_l_0110_0111: 502 Server Error: Bad Gateway for url: https://geo
   - ijssel_l_9517_9518: 502 Server Error: Bad Gateway for url: https://geo
   ... and 10 more

📊 Summary:

  land_use:
    - BrpGewas: 209 regions, 468 total features

  building_location:
    - bag:pand: 19 regions, 89 total features

  vegetation:
    - rws_vegetatielegger:bomen: 97 regions, 278 total features
    - rws_vegetatielegger:heggen: 7 regions, 20 total features
    - rws_vegetatielegger:vegetatieklassen: 218 regions, 1686 total features

  rws_legger:
    - rws_legger:andere_dan_primaire_waterkering_be

In [18]:
# Define output path
from pathlib import Path
import datetime

output_gpkg_path = Path("../data/all_wfs_data_batch_" + datetime.datetime.now().strftime("%Y%m%d_%H%M") + ".gpkg")

print(f"💾 Saving ALL WFS data to GeoPackage: {output_gpkg_path.name}\n")
print(f"This includes:")
print(f"  - Old services: {len(CONFIG.KNOWN_WFS_SERVICES)} services")
print(f"  - New services: {len(new_wfs_services)} services (RWS Legger + BKN Nature)")
print(f"  - Total regions processed: {len(all_region_ids)}")
print(f"  - Successful: {len(all_successful)}")
print(f"  - Failed: {len(all_failed)}\n")

saved_all_layers = []
service_summary = {}

for service_name, layers_dict in all_data_collection.items():
    print(f"\n📡 Service: {service_name}")
    service_layer_count = 0
    
    for layer_name, gdf_list in layers_dict.items():
        # Count how many non-empty GeoDataFrames we have
        non_empty_count = sum(1 for gdf in gdf_list if gdf is not None and len(gdf) > 0)
        
        if non_empty_count > 0:
            layer_output_name = save_wfs_layer_to_gpkg(
                wfs_data_list=gdf_list,
                service_name=service_name,
                layer_name=layer_name,
                output_path=output_gpkg_path,
                scope_region_ids=all_region_ids[:len(gdf_list)],  # Use all_region_ids
            )
            if layer_output_name:
                saved_all_layers.append(layer_output_name)
                service_layer_count += 1
                print(f"   ✅ {layer_name}: {non_empty_count} regions with data")
        else:
            print(f"   ⚠️  {layer_name}: No data (skipped)")
    
    service_summary[service_name] = service_layer_count

print(f"\n\n{'='*80}")
print(f"🎉 DONE! Saved {len(saved_all_layers)} layers to {output_gpkg_path.name}")
print(f"{'='*80}")

print(f"\n📊 Summary by Service:")
for service, count in service_summary.items():
    print(f"  {service:30s}: {count:3d} layers")

print(f"\n📋 All Saved Layers ({len(saved_all_layers)}):")
for layer in saved_all_layers:
    print(f"  - {layer}")

print(f"\n👉 Open {output_gpkg_path.name} in QGIS to explore!")
print(f"📂 Location: {output_gpkg_path.absolute()}")

💾 Saving ALL WFS data to GeoPackage: all_wfs_data_batch_20260114_1245.gpkg

This includes:
  - Old services: 3 services
  - New services: 2 services (RWS Legger + BKN Nature)
  - Total regions processed: 233
  - Successful: 218
  - Failed: 15


📡 Service: land_use
✅ Saved 205 features to layer: wfs_land_use_BrpGewas
   🔄 Removed 263 duplicates (56.2%)
   ✅ BrpGewas: 209 regions with data

📡 Service: building_location
✅ Saved 78 features to layer: wfs_building_location_bag_pand
   🔄 Removed 11 duplicates (12.4%)
   ✅ bag:pand: 19 regions with data

📡 Service: vegetation
✅ Saved 218 features to layer: wfs_vegetation_rws_vegetatielegger_bomen
   🔄 Removed 60 duplicates (21.6%)
   ✅ rws_vegetatielegger:bomen: 97 regions with data
✅ Saved 18 features to layer: wfs_vegetation_rws_vegetatielegger_heggen
   🔄 Removed 2 duplicates (10.0%)
   ✅ rws_vegetatielegger:heggen: 7 regions with data
✅ Saved 574 features to layer: wfs_vegetation_rws_vegetatielegger_vegetatieklassen
   🔄 Removed 1112 dupl

/var/folders/t3/519_jjmx4tz2_jlt9nm2gkpr0000gn/T/ipykernel_23315/3087777716.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(wfs_data_list, ignore_index=True)


✅ Saved 4 features to layer: wfs_rws_legger_rws_legger_genormeerd_bodem_legger
   🔄 Removed 238 duplicates (98.3%)
   ✅ rws_legger:genormeerd_bodem_legger: 217 regions with data
   ⚠️  rws_legger:in_of_uitwateringssluis_legger: No data (skipped)
✅ Saved 14 features to layer: wfs_rws_legger_rws_legger_kilometrering_legger
   🔄 Removed 7 duplicates (33.3%)
   ✅ rws_legger:kilometrering_legger: 21 regions with data
✅ Saved 118 features to layer: wfs_rws_legger_rws_legger_krib_legger
   🔄 Removed 60 duplicates (33.7%)
   ✅ rws_legger:krib_legger: 113 regions with data
✅ Saved 108 features to layer: wfs_rws_legger_rws_legger_kribhoogtes_legger
   🔄 Removed 43 duplicates (28.5%)
   ✅ rws_legger:kribhoogtes_legger: 105 regions with data
   ⚠️  rws_legger:kunstwerk_niet_in_beheer_bij_rws_legger: No data (skipped)
   ⚠️  rws_legger:lengteprofiel_over_primaire_kering_legger: No data (skipped)
   ⚠️  rws_legger:lengteprofiel_over_regionale_kering_legger: No data (skipped)
✅ Saved 15 features to l

/var/folders/t3/519_jjmx4tz2_jlt9nm2gkpr0000gn/T/ipykernel_23315/3087777716.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(wfs_data_list, ignore_index=True)


✅ Saved 10 features to layer: wfs_bkn_nature_beheerkaart_nat_beheer_vlakken
   🔄 Removed 263 duplicates (96.3%)
   ✅ beheerkaart_nat:beheer_vlakken: 149 regions with data
   ⚠️  beheerkaart_nat:exploitatie_vlakken: No data (skipped)
   ⚠️  beheerkaart_nat:gebouw_en_installatie_vlakken: No data (skipped)
   ⚠️  beheerkaart_nat:groen_lijnen: No data (skipped)
✅ Saved 122 features to layer: wfs_bkn_nature_beheerkaart_nat_groen_punten
   🔄 Removed 46 duplicates (27.4%)
   ✅ beheerkaart_nat:groen_punten: 45 regions with data
✅ Saved 1 features to layer: wfs_bkn_nature_beheerkaart_nat_groen_vlakken
   🔄 Removed 3 duplicates (75.0%)
   ✅ beheerkaart_nat:groen_vlakken: 4 regions with data
✅ Saved 1 features to layer: wfs_bkn_nature_beheerkaart_nat_kunstwerk_lijnen
   ✅ beheerkaart_nat:kunstwerk_lijnen: 1 regions with data
✅ Saved 14 features to layer: wfs_bkn_nature_beheerkaart_nat_kunstwerk_punten
   🔄 Removed 10 duplicates (41.7%)
   ✅ beheerkaart_nat:kunstwerk_punten: 8 regions with data
✅ 

In [ ]:
### 8.5 Visualize Downloaded Data on Map

import folium

# Load one of the downloaded layers to visualize
# Let's check what layers are in the output geopackage
import fiona
layers_in_gpkg = fiona.listlayers(output_gpkg)

print(f"📦 Layers in {output_gpkg.name}:")
for layer in layers_in_gpkg:
    print(f"   - {layer}")

# Pick a layer with data to visualize (e.g., BKN Nature exploitatie_vlakken)
if layers_in_gpkg:
    # Create map centered on scope regions
    center_lat = all_scope_regions.to_crs(epsg=4326).geometry.centroid.y.mean()
    center_lon = all_scope_regions.to_crs(epsg=4326).geometry.centroid.x.mean()
    
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=10,
        tiles='OpenStreetMap'
    )
    
    # Add scope regions as context
    folium.GeoJson(
        all_scope_regions.to_crs(epsg=4326),
        name="Scope Regions",
        style_function=lambda x: {
            'fillColor': 'lightblue',
            'color': 'blue',
            'weight': 1,
            'fillOpacity': 0.1
        }
    ).add_to(m)
    
    # Add first layer with data
    try:
        first_layer = layers_in_gpkg[0]
        data_layer = gpd.read_file(output_gpkg, layer=first_layer)
        
        if not data_layer.empty:
            folium.GeoJson(
                data_layer.to_crs(epsg=4326),
                name=first_layer,
                style_function=lambda x: {
                    'fillColor': 'green',
                    'color': 'darkgreen',
                    'weight': 2,
                    'fillOpacity': 0.5
                }
            ).add_to(m)
            
            print(f"\n🗺️  Visualizing: {first_layer} ({len(data_layer)} features)")
    except Exception as e:
        print(f"⚠️  Could not visualize: {e}")
    
    folium.LayerControl().add_to(m)
    display(m)
else:
    print("⚠️  No layers to visualize")

### 8.6 Next Steps & Conclusions

**✅ What We Accomplished:**
- Downloaded WFS data from RWS Legger and BKN Nature for all 233 scope regions
- Saved all results to geopackage: `wfs_new_sources_YYYYMMDD.gpkg`
- Identified which layers have data and coverage statistics

**📊 Key Findings:**
1. **RWS Legger:** Check coverage % - did we find infrastructure data?
2. **BKN Nature:** Check if nature management zones correlate with our regions
3. **Data Quality:** Review feature counts - are they meaningful?

**🔍 Next Actions:**

1. **Open in QGIS:**
   - Load the new geopackage
   - Overlay with existing `phase1_2025-08-14_v1.gpkg` layers
   - Visually assess which layers are useful for erosion prediction

2. **Prioritize Layers:**
   - Which have good coverage in our regions?
   - Which show features relevant to erosion (shores, structures, management zones)?
   - Which can be integrated into `DataHandler` feature extraction?

3. **Update Configuration:**
   - Add useful layers to `backend/src/config.py` → `KNOWN_WFS_SERVICES`
   - Define feature extraction in `AGGREGATION_COLUMNS`
   - Re-run `DataHandler.create_data_from_remote()` with new config

4. **Update Data Inventory:**
   - Mark RWS Legger and BKN Nature as "✅ In Use" or "❌ Skip" based on findings
   - Document which specific layers are valuable

5. **Stakeholder Communication:**
   - Report findings to Pam/Etienne
   - Include coverage statistics and visual examples
   - Identify action items (e.g., still need Vegetatiemonitor access, AIS data, Waterweb endpoints)

**📧 Email Summary Points:**
- "Tested RWS Legger and BKN Nature across all 233 regions"
- "RWS Legger: [X% coverage, Y features in layers A, B, C]"
- "BKN Nature: [X% coverage, Y features in layers D, E, F]"
- "Next: Visual assessment + integration into model pipeline"